# 036 · AdaGrad (Adaptive Gradient)

The first attempt at lesson 032's **problem 2**: one learning rate for every
weight, when different parameters want different step sizes.

$$w \leftarrow w - \frac{\eta}{\sqrt{G_t} + \varepsilon}\nabla L, \qquad G_t = \sum_{i \le t} g_i^2$$

| Part | What we reproduce |
|---|---|
| A | sparse features: a column non-zero in **2%** of rows gets a gradient **11.7× smaller** |
| B | plain SGD recovers **66%** of the sparse weight; AdaGrad reaches **99%** |
| C | why squared, and why the square root |
| D | the fatal flaw — `G_t` only ever grows, so the rate only ever shrinks |

Needs `numpy`.

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
N = 500

dense = rng.normal(0, 1, N)
sparse = (rng.random(N) < 0.04).astype(float)   # "went to a top-5 college?" - rare
X = np.c_[dense, sparse]
TRUE_W = np.array([1.0, 5.0])                   # the RARE feature matters MORE
y = X @ TRUE_W + rng.normal(0, 0.1, N)

def grads(w):
    return 2 * X.T @ (X @ w - y) / N

def mse(w):
    return float(np.mean((X @ w - y) ** 2))

print(f"sparse feature non-zero in {int(sparse.sum())}/{N} rows ({sparse.mean():.0%})")

## Part A — Why a sparse feature gets starved

For a linear model, `∂L/∂w ∝ x`. When `x = 0` the gradient for that weight is
**exactly zero** — not small, zero. So a feature that is present in 2% of rows
receives a useful gradient in 2% of rows.

In [ ]:
g0 = np.abs(grads(np.zeros(2)))
print(f"gradient at the start:")
print(f"  dense  parameter : {g0[0]:.3f}")
print(f"  sparse parameter : {g0[1]:.3f}")
print(f"  -> {g0[0]/g0[1]:.1f}x smaller for the sparse parameter")
print("\nAnd note this has NOTHING to do with importance: the sparse feature")
print(f"has the larger true weight ({TRUE_W[1]} against {TRUE_W[0]}).")

assert abs(g0[0]/g0[1] - 11.7) < 0.2

In [ ]:
# The gradient is zero on most individual rows. Look at them one at a time.
per_row = 2 * X * (X @ np.zeros(2) - y)[:, None]
zero_rows = (per_row[:, 1] == 0).mean()
print(f"rows where the sparse parameter's gradient is EXACTLY zero: {zero_rows:.0%}")
print("With one learning rate, that parameter simply moves less. Forever.")

## Part B — What AdaGrad does about it

Divide each parameter's step by the square root of its **accumulated** squared
gradients. A parameter that has received small gradients gets a proportionally
larger step.

In [ ]:
def train(adaptive, lr, steps=300):
    w, acc = np.zeros(2), np.zeros(2)
    for _ in range(steps):
        g = grads(w)
        if adaptive:
            acc += g ** 2                                  # a SUM
            w = w - lr * g / (np.sqrt(acc) + 1e-8)
        else:
            w = w - lr * g
    return w


for name, adaptive, lr in (("plain SGD", False, 0.1), ("AdaGrad", True, 0.5)):
    w = train(adaptive, lr)
    print(f"  {name:<10} w = [{w[0]:.3f}, {w[1]:.3f}]   sparse weight at "
          f"{w[1]/TRUE_W[1]:.0%} of true    mse = {mse(w):.4f}")
print(f"  {'true':<10} w = [{TRUE_W[0]:.3f}, {TRUE_W[1]:.3f}]")

w_sgd, w_ada = train(False, 0.1), train(True, 0.5)
print(f"\nAdaGrad's MSE is {mse(w_sgd)/mse(w_ada):.0f}x lower.")
assert w_ada[1] / TRUE_W[1] > 0.95
assert w_sgd[1] / TRUE_W[1] < 0.75

Both recover the dense weight almost perfectly. The difference is entirely in
the sparse one — which is the parameter that mattered most.

## Part C — Why squared, and why the square root

In [ ]:
g = np.array([3.0, -3.0, 3.0, -3.0])
print("a parameter whose gradient keeps flipping sign:")
print(f"  plain sum of gradients : {g.sum():.1f}   <- cancels to nothing")
print(f"  sum of SQUARED         : {(g**2).sum():.1f}   <- measures activity")
print("\nSquaring is what makes G_t a measure of how much a parameter has been")
print("moving, regardless of direction.")

In [ ]:
# And the square root restores the units.
print("if every gradient doubles, the step should stay comparable:")
for scale in (1.0, 2.0, 10.0):
    g = scale * np.ones(50)
    acc = (g ** 2).sum()
    step = 0.5 * g[0] / (np.sqrt(acc) + 1e-8)
    print(f"  gradients x{scale:<5} -> accumulated {acc:>8.1f}, step {step:.5f}")

print("\nWith the square root, the step is scale-invariant. Without it, the")
print("step would shrink with the SQUARE of the gradient scale.")

## Part D — The fatal flaw

`G_t` is a **sum of squares**. Squares are non-negative, so a sum of them can
only ever grow — which means the effective learning rate can only ever shrink,
and there is no way back.

In [ ]:
rng2 = np.random.default_rng(7)
stream = rng2.normal(0.0, 1.0, 1000)      # gradient noise that never vanishes

acc, lr, eps = 0.0, 0.1, 1e-8
print(f"{'step':>6}{'accumulator':>14}{'effective lr':>15}")
for t, g in enumerate(stream, 1):
    acc += g ** 2
    if t in (10, 100, 500, 1000):
        print(f"{t:>6}{acc:>14.1f}{lr/(np.sqrt(acc)+eps):>15.5f}")

print("\nThe accumulator has grown ~190x and the rate has fallen accordingly.")
print("It is STILL FALLING at step 1000, and it cannot recover, because G_t")
print("cannot decrease. Training stalls before it has converged.")

In [ ]:
# See the stall directly: give AdaGrad a much longer run and watch it stop moving.
def trace(steps):
    w, acc = np.zeros(2), np.zeros(2)
    marks = {}
    for t in range(1, steps + 1):
        g = grads(w)
        acc += g ** 2
        w = w - 0.5 * g / (np.sqrt(acc) + 1e-8)
        if t in (50, 200, 1000, 3000):
            marks[t] = (w.copy(), mse(w))
    return marks

for t, (w, m) in trace(3000).items():
    print(f"  step {t:>5}: sparse weight {w[1]:.4f}  mse {m:.5f}")

print("\nThe last stretch buys almost nothing. That is the stall.")
print("\nThe fix is to replace the SUM with an EWMA, so old gradients fade.")
print("That is RMSprop, and it is a one-line change.")

## What to take away

- **AdaGrad gives every parameter its own learning rate** — lesson 032's
  problem 2, first attempt.
- **`w ← w − η/(√G_t + ε) ∇L`**, where `G_t` is that parameter's **accumulated
  squared gradients**.
- **Sparse features are the motivating case:** `∂L/∂w ∝ x`, so when `x = 0` the
  gradient is **exactly zero**.
- Measured: a feature non-zero in **2%** of rows had a gradient **11.7× smaller**
  — not because it mattered less. Its true weight was the *larger* of the two.
- After 300 steps, plain SGD recovered **66%** of the sparse weight; **AdaGrad
  reached 99%**, with 6× lower MSE.
- **Squared** so sign does not matter; **square root** to restore the units.
- **The fatal flaw: `G_t` is a sum of squares, so it only ever grows** — the
  effective rate only ever shrinks.
- **Training therefore stalls before converging**, with no way back.
- **Fix: replace the sum with an EWMA** so old gradients fade — that is **RMSprop**.

## Exercises

1. Change the sparsity from 4% to 20% and to 1%. How does AdaGrad's advantage
   scale with rarity?
2. Part B gave the two methods different learning rates (0.1 and 0.5). Is that
   fair? Tune each one properly and see whether the conclusion survives.
3. AdaGrad's stall depends on the run length. Find the step count at which plain
   SGD overtakes AdaGrad on this problem, if it ever does.
4. Add ε inside the square root rather than outside. Does it matter? Try
   ε = 1e-8 and ε = 1e-2.
5. Implement AdaGrad on the ravine from lesson 034. Does per-parameter scaling
   solve the oscillation problem too, or only the step-size one?